Concatenate posts by author: group all posts by each author into one document per author

In [1]:
import pandas as pd 
import matplotlib.pyplot as plt
import re
import numpy as np
from datetime import datetime, timedelta
import os
import seaborn as sns


In [2]:
# Load (unprocessed) data 
df = pd.read_csv("../data/unfiltered/endo_and_ibs_all_posts.csv")

In [ ]:
# Examine the title and selftext columns
print("Sample of the 'title' and 'selftext' columns:")
for i, row in enumerate(df[['title', 'selftext']].head(3).itertuples(index=False)):
    print(f"\nPost {i+1}:")
    print(f"Author: {df.iloc[i]['author']}")
    print(f"Subreddit: {df.iloc[i]['subreddit']}")
    print(f"Label: {df.iloc[i]['label']}")
    print(f"Title: {row[0]}")
    print(f"Selftext: {row[1]}...")
    
print(f"\nTotal authors in dataset: {df['author'].nunique()}")
print(f"Posts per author statistics:")
posts_per_author = df['author'].value_counts()
print(posts_per_author.describe())

In [ ]:
# Create author documents by concatenating all posts (title + selftext) for each author
author_documents = []

for author in df['author'].unique():
    author_data = df[df['author'] == author].copy()
    
    # Sort chronologically
    author_data = author_data.sort_values('created_utc')
    
    # Get label
    author_label = author_data['label'].iloc[0]
    
    # Check for consistency
    if len(author_data['label'].unique()) > 1:
        print(f"Warning: Author {author} has mixed labels: {author_data['label'].unique()}")
    
    # Combine title + selftext per post
    posts = (
        author_data['title'].fillna('') + ' ' +
        author_data['selftext'].fillna('')
    )
    
    # Concatenate posts
    concatenated_text = " ".join(posts.astype(str))
    
    # Metadata
    post_count = len(author_data)
    subreddits = list(author_data['subreddit'].unique())
    
    author_documents.append({
        'author': author,
        'label': author_label,
        'post_count': post_count,
        'subreddits': subreddits,
        'subreddit_count': len(subreddits),
        'concatenated_text': concatenated_text,
        'text_length': len(concatenated_text)
    })

# Create DataFrame
author_df = pd.DataFrame(author_documents)

print(f"Created {len(author_df)} author documents")

print("\nAuthor document statistics:")
print(author_df[['post_count', 'subreddit_count', 'text_length']].describe())

print(f"\nAuthors by condition:")
label_counts = author_df['label'].value_counts().sort_index()
for label, count in label_counts.items():
    condition = "Endometriosis" if label == 1 else "IBS"
    print(f"  {condition} (Label {label}): {count} authors")

In [ ]:
# Show some examples of concatenated documents
for i in range(2):
    sample = author_df.iloc[i]
    condition = "Endometriosis" if sample['label'] == 1 else "IBS"
    
    print(f"\nAuthor: {sample['author']}")
    print(f"Condition: {condition} (Label {sample['label']})")
    print(f"Number of posts: {sample['post_count']}")
    print(f"Number of unique subreddits: {sample['subreddit_count']}")
    print(f"Total text length: {sample['text_length']:,} characters")
    print(f"Subreddits: {', '.join(sample['subreddits'][:5])}{'...' if len(sample['subreddits']) > 5 else ''}")
    print(f"First 300 characters of concatenated text:")
    print(f'"{sample["concatenated_text"][:300]}..."')
    print("-" * 60)

# Analysis by condition
print("\nAnalysis by condition:")

for label in sorted(author_df['label'].unique()):
    condition = "Endometriosis" if label == 1 else "IBS"
    condition_data = author_df[author_df['label'] == label]
    
    print(f"\n{condition} (Label {label}):")
    print(f"  Authors: {len(condition_data)}")
    print(f"  Avg posts per author: {condition_data['post_count'].mean():.1f}")
    print(f"  Avg subreddits per author: {condition_data['subreddit_count'].mean():.1f}")
    print(f"  Avg text length per author: {condition_data['text_length'].mean():,.0f} characters")
    print(f"  Median text length per author: {condition_data['text_length'].median():,.0f} characters")

In [ ]:
# Save the concatenated author documents to CSV files
author_metadata = author_df.drop('concatenated_text', axis=1).copy()

author_metadata['subreddits'] = author_metadata['subreddits'].apply(
    lambda x: ', '.join([str(s) for s in x if pd.notna(s)])
)

# Save metadata
author_metadata.to_csv('../data/unfiltered/concatenated/author_metadata.csv', index=False)

# Save full documents
author_df_export = author_df.copy()
author_df_export['subreddits'] = author_df_export['subreddits'].apply(
    lambda x: ', '.join([str(s) for s in x if pd.notna(s)])
)
author_df_export.to_csv('../data/unfiltered/concatenated/author_documents_full.csv', index=False)

# Create separate files for each condition
for label in sorted(author_df['label'].unique()):
    condition = "Endometriosis" if label == 1 else "IBS"
    condition_data = author_df_export[author_df_export['label'] == label]
    filename = f"../data/unfiltered/concatenated/author_documents_{condition.lower()}.csv"
    condition_data.to_csv(filename, index=False)
